In [1]:


try:
    import google.colab  # type: ignore
    from google.colab import output

    COLAB = True
    %pip install sae-lens transformer-lens sae-dashboard
except:
    COLAB = False
    from IPython import get_ipython  # type: ignore

    ipython = get_ipython()
    assert ipython is not None
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

# Standard imports
import os
import torch
from tqdm import tqdm
import plotly.express as px

# Imports for displaying vis in Colab / notebook
import webbrowser
import http.server
import socketserver
import threading

PORT = 8000

torch.set_grad_enabled(False)

# For the most part I'll try to import functions and classes near where they are used
# to make it clear where they come from.

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

def display_vis_inline(filename: str, height: int = 850):
    """
    Displays the HTML files in Colab. Uses global `PORT` variable defined in prev cell, so that each
    vis has a unique port without having to define a port within the function.
    """
    if not (COLAB):
        webbrowser.open(filename)

    else:
        global PORT

        def serve(directory):
            os.chdir(directory)

            # Create a handler for serving files
            handler = http.server.SimpleHTTPRequestHandler

            # Create a socket server with the handler
            with socketserver.TCPServer(("", PORT), handler) as httpd:
                print(f"Serving files from {directory} on port {PORT}")
                httpd.serve_forever()

        thread = threading.Thread(target=serve, args=("/content",))
        thread.start()

        output.serve_kernel_port_as_iframe(
            PORT, path=f"/{filename}", height=height, cache_in_notebook=True
        )

        PORT += 1

Device: cuda


In [2]:
from sae_lens import SAE
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer




model = HookedTransformer.from_pretrained("gemma-2-2b", device=device)
release = "gemma-scope-2b-pt-res-canonical"
sae_id = "layer_1/width_16k/canonical"
#sae = SAE.from_pretrained(release, sae_id,device=device)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


In [3]:
sae = SAE.from_pretrained(release, sae_id,device="cpu")
out,cache=model.run_with_cache("The capital of Poland is")
feature_acts = sae.encode(cache[sae.cfg.metadata.hook_name].to("cpu"))
sae_out = sae.decode(feature_acts)
#model.blocks czy coś żeby wyświetlić aktywacje
print(model)

/home/mm/projects/expai/circuit-tracer/lib/python3.10/site-packages/sae_lens/saes/sae.py:249: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-25): 26 x TransformerBlock(
      (ln1): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln1_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (

# Expreiments

In [ ]:
print(torch.max(feature_acts))
print(model.blocks[1].attn.W_Q.shape)
print(model.blocks[1].attn.W_K.shape)
print(torch.min(model.blocks[1].attn.b_Q))
print(torch.max(model.blocks[1].attn.b_K))
print(sae)

In [ ]:
from IPython.display import IFrame
html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

def get_dashboard_html(sae_release = "gemma-2-2b", sae_id="20-gemmascope-res-16k", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)

html = get_dashboard_html(sae_release = "gemma-2-2b", sae_id="1-gemmascope-res-16k", feature_idx=2009)
IFrame(html, width=1200, height=600)

In [ ]:
Neuronpedia_API_KEY="sk-np-s0NXleMtqsfhvva3S0ZFZL0O7QEl0V6NYhkgknTWiUo0"

In [ ]:
import requests
res=requests.get(
    "https://www.neuronpedia.org/api/feature/gemma-2-2b/1-gemmascope-res-16k/2",
    headers={
      "x-api-key": Neuronpedia_API_KEY
    }
)
data=res.json()
print(data)
print(data.get("explanations"))

In [ ]:
#print(data.get("explanations")[0]["description"])
#print(feature_acts.shape)
def get_label(idx):
    res=requests.get(
        f"https://www.neuronpedia.org/api/feature/gemma-2-2b/1-gemmascope-res-16k/{idx}",
        headers={
          "x-api-key": Neuronpedia_API_KEY
        }
    )
    data=res.json().get("explanations")
    if len(data)==0:
        return "unknown"
    else:  
      return data[0]["description"]

In [ ]:
from tqdm import tqdm
#lepiej tu dodawać w pętli, przynajmniej jak się wywali to się nie straci wszystkiego
#labels=[get_label(i) for i in range(feature_acts.shape[2])]
#labels=[]
for i in tqdm(range(6729,feature_acts.shape[2])):
    labels.append(get_label(i))

 

In [ ]:
import json
with open('labels_final.json', 'w') as f:
    json.dump(labels, f)

In [ ]:
print(feature_acts.shape)
print(feature_acts[0].shape)
#print(sae_out.shape)
print(sae.W_dec.shape)
#feature_acts@sae_out
print((feature_acts@sae.W_dec).shape)
#spr czy dobrze mnoży
print((sae.W_dec*feature_acts[0][1].reshape(-1,1)).shape)

print((model.blocks[1].attn.W_Q[0,:,:]@model.blocks[1].attn.W_K[0,:,:].T).shape)
#print(model.blocks[1].attn.W_K.shape)

In [ ]:
Features=sae.W_dec*feature_acts[0][1].reshape(-1,1)
W_QK=(model.blocks[1].attn.W_Q[0,:,:]@model.blocks[1].attn.W_K[0,:,:].T).to("cpu")
print((Features@W_QK@Features.T).shape)

In [ ]:
print(Features.shape)
print(len(labels))
print(get_label(6728))
print(labels[-1])

In [ ]:
for i in range(1,2):
    print(i)

print(out)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")
input_ids = tokenizer("The capital of Poland is", return_tensors="pt").to("cpu")
print(input_ids)

In [ ]:
for i in range(input_ids['input_ids'].shape[1]):
    print(tokenizer.decode(input_ids['input_ids'][0][i]))

In [ ]:
print(labels[2202])

# Visualisation

In [4]:
from transformers import AutoTokenizer
import pandas as pd

def get_feature_interactions(activations,x,y,head):
    W_QK=(model.blocks[1].attn.W_Q[head,:,:]@model.blocks[1].attn.W_K[head,:,:].T).to("cpu")
    features_x=sae.W_dec*activations[0][x].reshape(-1,1)
    features_y=sae.W_dec*activations[0][y].reshape(-1,1)
    feature_interactions=(features_x@W_QK@features_y.T)
    return feature_interactions

def get_active_features(feature_interactions,labels):
    nonzero_mask = ((feature_interactions*feature_interactions).sum(dim=0) != 0) | ((feature_interactions*feature_interactions).sum(dim=1) != 0)
    features_pruned = feature_interactions[nonzero_mask][:, nonzero_mask]
    labels_pruned = [l for l, keep in zip(labels, nonzero_mask.tolist()) if keep]
    return features_pruned, labels_pruned

def plot_feature_interactions(feature_interactions,tokens,labels,x,y,head):
    l=len(labels)
    df = pd.DataFrame(feature_interactions.numpy()[:l,:l], index=labels, columns=labels)
    fig = px.imshow(df, text_auto=True, aspect="auto", title="Head {} Feature Interactions between {} and {}".format(head,tokens[x],tokens[y]))
    fig.update_layout(width=800, height=800)
    fig.show()

def get_tokens(input):
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")
    input_ids = tokenizer("The capital of Poland is", return_tensors="pt").to("cpu")
    tokens = [tokenizer.decode(input_ids['input_ids'][0][i]) for i in range(input_ids['input_ids'].shape[1])]
    return tokens


In [5]:
import json
with open("labels_final.json", "r") as f:
    labels = json.load(f)


In [22]:
def plot_feature_interactions_new(
    feature_interactions, tokens, labels, x, y, head,
    cell_size=20, max_size=2000, font_size=10, numpy=False
):
    l = len(labels)
    if numpy:
        df = pd.DataFrame(
        feature_interactions[:l, :l],
        index=labels,
        columns=labels)
    else:
        df = pd.DataFrame(
            feature_interactions.numpy()[:l, :l],
            index=labels,
            columns=labels
        )
        
    # Size of the square grid itself
    grid_size = min(cell_size * l, max_size)

    # Total figure size = grid_size + margins
    extra_left = 300  # space for y-axis labels
    extra_bottom = 300 # space for x-axis labels
    extra_top = 100
    extra_right = 50
    
    fig = px.imshow(
        df,
        text_auto=(l <= 30),  # only show text when grid is small
        aspect="equal",       # keep squares square
        title=f"Head {head} Feature Interactions between {tokens[x]} and {tokens[y]}"
    )
    
    fig.update_layout(
        width=grid_size + extra_left + extra_right,
        height=grid_size + extra_top + extra_bottom,
        xaxis=dict(
            tickfont=dict(size=font_size),
            automargin=False
        ),
        yaxis=dict(
            tickfont=dict(size=font_size),
            automargin=False
        ),
        margin=dict(
            l=extra_left, r=extra_right,
            t=extra_top, b=extra_bottom
        )
    )
    
    fig.show()

In [7]:
input="The capital of Poland is"
head=0
interactions=[]
for i in range(1,feature_acts[0].shape[0]):
    for j in range(i,feature_acts[0].shape[0]):
        feature_interactions=get_feature_interactions(feature_acts,i,j,head)
        features_pruned, labels_pruned = get_active_features(feature_interactions,labels)
        tokens=get_tokens(input)
        print(features_pruned.shape,len(labels_pruned))
        interactions.append((i,j,features_pruned,labels_pruned))
        plot_feature_interactions_new(features_pruned,tokens,labels_pruned,i,j,head)

torch.Size([28, 28]) 28


torch.Size([99, 99]) 99


torch.Size([56, 56]) 56


torch.Size([168, 168]) 168


torch.Size([51, 51]) 51


...

torch.Size([173, 173]) 173


torch.Size([56, 56]) 56


torch.Size([144, 144]) 144


torch.Size([168, 168]) 168


torch.Size([29, 29]) 29


# More experiments

In [ ]:
def is_symmetric(tensor, tol=1e-8):
    # Check it's a 2D square matrix
    if tensor.dim() != 2 or tensor.size(0) != tensor.size(1):
        return False
    
    # Check if equal to transpose (within a tolerance for floating-point numbers)
    return torch.allclose(tensor, tensor.T, atol=tol)

feature_interactions=Features@W_QK@Features.T
print(is_symmetric(feature_interactions))
print(torch.argmax(feature_interactions))

# 1. Find which rows/columns are nonzero
nonzero_mask = ((feature_interactions*feature_interactions).sum(dim=0) != 0) | ((feature_interactions*feature_interactions).sum(dim=1) != 0)

# 2. Filter rows/cols
T_pruned = feature_interactions[nonzero_mask][:, nonzero_mask]

# 3. Filter labels accordingly
labels_pruned = [l for l, keep in zip(labels, nonzero_mask.tolist()) if keep]

In [ ]:
print(T_pruned.shape)
print(len(labels_pruned))

In [ ]:
import pandas as pd


#trzeba wyświetlić tylko niezerowe
df = pd.DataFrame(T_pruned.numpy()[:len(labels_pruned),:len(labels_pruned)], index=labels_pruned, columns=labels_pruned)
fig = px.imshow(df, text_auto=True, aspect="auto", title="Feature Interactions")
fig.update_layout(width=800, height=800)
fig.show()

In [ ]:
import json
with open("labels4.json", "r") as f:
    labels = json.load(f)

labels.append(labels[-1])

for i in range(len(labels)-1,2007,-1):
    labels[i]=labels[i-1]
    #print(i)

labels[2007]="unknown"

for i in range(2007,len(labels)):
    labels[i]=str(i)+": "+labels[i]


l=len(labels)

In [ ]:
print(l)
print(labels[-1])
print(labels[2009])

In [ ]:
feature_interactions=Features@W_QK@Features.T
import pandas as pd


#trzeba wyświetlić tylko niezerowe
df = pd.DataFrame(feature_interactions.numpy()[:l,:l], index=labels, columns=labels)
fig = px.imshow(df, text_auto=True, aspect="auto", title="Feature Interactions")
fig.update_layout(width=800, height=800)
fig.show()

In [ ]:
print(feature_interactions)

In [ ]:
print(labels)

In [ ]:
print(cache)

In [ ]:
print(type(sae.cfg.metadata.hook_name))
print(type('ziu'))

In [ ]:
# Creating a small non-symmetric 3x3 interaction matrix, performing truncated SVD (k=2),
# and showing the left/right singular vectors and the row/column embeddings.
import numpy as np
import pandas as pd

# matrix (non-symmetric)
M = np.array([
    [0.0, 1.2, 2.5],
    [3.1, 0.0, 4.2],
    [5.3, 6.7, 0.0]
])

# full SVD
U, s, Vt = np.linalg.svd(M, full_matrices=False)
V = Vt.T

# truncate to k=2
k = 2
U_k = U[:, :k]
S_k = np.diag(s[:k])
V_k = V[:, :k]

# row and column coordinates (rows: U_k @ S_k, columns: V_k @ S_k)
row_coords = U_k @ S_k    # n x k
col_coords = V_k @ S_k    # n x k

# low-rank reconstruction
M_approx = U_k @ S_k @ V_k.T

# Pack results as DataFrames for nicer display
df_M = pd.DataFrame(M, index=[f"f{i}" for i in range(1,4)], columns=[f"f{i}" for i in range(1,4)])
df_Uk = pd.DataFrame(U_k, index=[f"f{i}" for i in range(1,4)], columns=[f"u{j}" for j in range(1,k+1)])
df_Vk = pd.DataFrame(V_k, index=[f"f{i}" for i in range(1,4)], columns=[f"v{j}" for j in range(1,k+1)])
df_Sk = pd.DataFrame(S_k, index=[f"s{j}" for j in range(1,k+1)], columns=[f"s{j}" for j in range(1,k+1)])
df_row = pd.DataFrame(row_coords, index=[f"f{i}" for i in range(1,4)], columns=[f"dim{j}" for j in range(1,k+1)])
df_col = pd.DataFrame(col_coords, index=[f"f{i}" for i in range(1,4)], columns=[f"dim{j}" for j in range(1,k+1)])
df_approx = pd.DataFrame(M_approx, index=[f"f{i}" for i in range(1,4)], columns=[f"f{i}" for i in range(1,4)])

# Display summaries
print("Original matrix M:")
display(df_M)
print("\nTruncated left singular vectors U_k (columns are left singular vectors):")
display(df_Uk)
print("\nTruncated right singular vectors V_k (columns are right singular vectors):")
display(df_Vk)
print("\nTruncated singular values S_k:")
display(df_Sk)
print("\nRow coordinates (U_k @ S_k) — embedding for rows (features as sources):")
display(df_row)
print("\nColumn coordinates (V_k @ S_k) — embedding for columns (features as targets):")
display(df_col)
print("\nLow-rank reconstruction M_approx = U_k S_k V_k^T:")
display(df_approx)
print("\nReconstruction error (Frobenius norm of M - M_approx):")
print(np.linalg.norm(M - M_approx, ord='fro'))


# Dim reduction

In [19]:
from cur import cur_decomposition
r=32
print(interactions[1])
i,j,M,labels_pruned=interactions[1]
print(tokens[i],tokens[j])
Large=M.numpy()
print(Large)
C, U, R = cur_decomposition(Large, r)
#print(C,U,R)

(1, 2, tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0256,  0.0000,  0.0043,  ...,  0.0370,  0.0000,  0.0059],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.0601,  0.0000, -0.7794,  ..., -0.7745,  0.0000, -0.0546],
        [ 0.0360,  0.0000, -0.4190,  ..., -0.2056,  0.0000,  0.0265],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]), ['17: keywords related to political events and actions involving election procedures and government officials', '181: technical issues related to software and system errors', '392: financial and medical terms related to conditions or processes', '705: references to significant amounts of money or economic transactions', '746:  references to product details and descriptions', '1245: phrases that indicate the existence or presence of something', '1247: key phrases related to research study results and purpose', '1512: references to water and its properti

In [20]:
print(C,U,R)

[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [-0.16025816 -0.06777783 -0.06777783 ...  0.0095277  -0.25586703
   0.01784465]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.6899532  -0.79590505 -0.79590505 ... -0.03434465  0.43096432
  -0.37344152]
 [-0.22847857 -0.16830866 -0.16830866 ...  0.09236307 -0.0045296
  -0.09912424]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]] [[-2.76401858e+28 -6.22765811e+28  8.17790716e+28 ...  1.54784062e+27
  -3.82774831e+12 -1.16371012e+27]
 [-4.31245297e+27 -7.92882794e+27 -5.30788503e+27 ...  2.32013304e+27
  -5.20349653e+12 -2.04256945e+27]
 [-4.84268087e+27 -4.28111328e+27  1.10016275e+28 ... -1.05642437e+26
   3.03483034e+12  7.35267798e+27]
 ...
 [ 4.02190170e+27  1.08695996e+28 -2.16715537e+28 ... -1.30312523e+27
   4.89130951e+11 -4.35321969e+27]
 [-2.45338125e+27 -4.56030283e+27  5.03531712e+27 ... -9.34081213e+26
   1.06717315e+12  6.48603283e+27]
 [-

In [23]:
plot_feature_interactions_new(C@U@R,tokens,labels_pruned,i,j,head,numpy=True)

In [24]:
plot_feature_interactions_new(M,tokens,labels_pruned,i,j,head)

In [29]:
import matplotlib.pyplot as plt
def plot_comb_interactions(C,U,R, tokens, labels, x, y, head,
    cell_size=20, max_size=2000, font_size=10, numpy=False
):
    l=U.shape[0]
    df = pd.DataFrame(feature_interactions[:l, :l])#,index=labels,columns=labels)

        
    # Size of the square grid itself
    grid_size = min(cell_size * l, max_size)

    # Total figure size = grid_size + margins
    extra_left = 300  # space for y-axis labels
    extra_bottom = 300 # space for x-axis labels
    extra_top = 100
    extra_right = 50
    
    fig = px.imshow(
        df,
        text_auto=(l <= 30),  # only show text when grid is small
        aspect="equal",       # keep squares square
        title=f"Head {head} Feature Interactions between {tokens[x]} and {tokens[y]}"
    )
    
    fig.update_layout(
        width=grid_size + extra_left + extra_right,
        height=grid_size + extra_top + extra_bottom,
        xaxis=dict(
            tickfont=dict(size=font_size),
            automargin=False
        ),
        yaxis=dict(
            tickfont=dict(size=font_size),
            automargin=False
        ),
        margin=dict(
            l=extra_left, r=extra_right,
            t=extra_top, b=extra_bottom
        )
    )
    
    fig.show()

def plot_heatmap(array, cmap="viridis", show_colorbar=True, title=None):
    """
    Plot a 2D numpy array as a heatmap.
    
    Parameters:
        array (np.ndarray): 2D numpy array to visualize.
        cmap (str): Colormap (default "viridis").
        show_colorbar (bool): Whether to display the colorbar (default True).
        title (str): Optional title for the plot.
    """
    if array.ndim != 2:
        raise ValueError("Input must be a 2D numpy array.")
    
    plt.figure(figsize=(6, 5))
    heatmap = plt.imshow(array, cmap=cmap, aspect="auto", origin="upper")
    
    if show_colorbar:
        plt.colorbar(heatmap, shrink=0.8)
    
    if title:
        plt.title(title)
    
    plt.xlabel("Columns")
    plt.ylabel("Rows")
    plt.tight_layout()
    plt.show()

In [27]:
print(C.shape)
print(U.shape)
print(R.shape)

(99, 32)
(32, 32)
(32, 99)


In [51]:
import numpy as np
def plot_comb_interactions_mod(
    C, U, R, tokens, labels, x, y, head,
    cell_size=20, max_size=2000, font_size=10,
    numpy=False, log_scale=True, clip_percentile=99
):
    l = U.shape[0]
    
    # Extract interaction matrix
    mat = C#[:l, :l] if numpy else C
    
    # Optionally log-scale values (preserves sign)
    if log_scale:
        mat = np.sign(mat) * np.log1p(np.abs(mat))
    
    # Clip extreme values for better contrast
    if clip_percentile is not None:
        lim = np.percentile(np.abs(mat), clip_percentile)
        mat = np.clip(mat, -lim, lim)
    
    df = pd.DataFrame(mat)  # optionally: index=labels, columns=labels
    
    # Size of the square grid itself
    grid_size = min(cell_size * l, max_size)

    # Total figure size = grid_size + margins
    extra_left = 300
    extra_bottom = 300
    extra_top = 100
    extra_right = 50
    
    fig = px.imshow(
        df,
        text_auto=(l <= 30),
        aspect="equal",
        color_continuous_scale="RdBu",  # diverging for positive/negative contrast
        zmin=-np.max(np.abs(mat)),      # center scale at 0
        zmax=np.max(np.abs(mat)),
        title=f"Head {head} Feature Interactions between {tokens[x]} and {tokens[y]}"
    )
    
    fig.update_layout(
        width=grid_size + extra_left + extra_right,
        height=grid_size + extra_top + extra_bottom,
        xaxis=dict(
            tickfont=dict(size=font_size),
            automargin=False
        ),
        yaxis=dict(
            tickfont=dict(size=font_size),
            automargin=False
        ),
        margin=dict(
            l=extra_left, r=extra_right,
            t=extra_top, b=extra_bottom
        )
    )
    
    fig.show()

In [52]:
#plot_feature_interactions_new(C,U,R,tokens,labels_pruned,i,j,head,numpy=True)
plot_comb_interactions_mod(U,U,U,tokens,labels_pruned,i,j,head,numpy=True)
plot_comb_interactions_mod(C,C,C,tokens,labels_pruned,i,j,head,numpy=True)
plot_comb_interactions_mod(R,R,R,tokens,labels_pruned,i,j,head,numpy=True)

In [48]:
print(R.shape)
plot_comb_interactions_mod(R,R,R,tokens,labels_pruned,i,j,head,numpy=True)
#print(U)

(32, 99)


In [44]:
np.set_printoptions(threshold=np.inf)
np.set_printoptions(linewidth=600)
print(C)

[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-1.60258159e-01 -6.77778274e-02 -6.77778274e-02  2.82601446e-01  2.82601446e-01  5.98803759e-02  2.82601446e-01  1.78446453e-02  2.82601446e-01  4.09234911e-02 -2.23693624e-01  2.95790602e-02  6.57580746e-03  2.82601446e-01  2.82601446e-01 -9.14156884e-02 -2.55867034e-01 -1.32576721e-02  6.96984082e-02  1.78446453e-02 -7.34800622e-02 -1.93581447e-01  6.57580746e-03  5.98803759e-02  5.98803759e-02  2.82601446e-01 -6.77778274e-02  2.82601446e-01  1.78446453e-02  9.52770468e-03 -2.

In [40]:
print(R)

[[-7.31765628e-02  0.00000000e+00 -6.73802078e-01  0.00000000e+00  9.12313312e-02  0.00000000e+00 -3.80682677e-01  1.26172593e-02  7.83083066e-02 -1.03872150e-01  1.28826620e-02 -4.72253747e-02  1.66898817e-01  3.61725129e-02  1.18976220e-01  0.00000000e+00  7.52194345e-01 -9.49485693e-05
   8.92559662e-02 -6.93785250e-01 -1.00850217e-01  8.33363608e-02  1.67155892e-01 -1.93190858e-01  3.17228943e-01  0.00000000e+00 -8.64206553e-02  8.51562321e-02 -1.15364760e-01  0.00000000e+00  1.18489780e-01  0.00000000e+00  1.65713672e-02 -2.60931045e-01 -1.44681856e-01  1.17454216e-01
   1.19546905e-01 -1.15434796e-01  0.00000000e+00 -1.80262085e-02 -3.81686181e-01  5.22242039e-02  5.51019497e-02  3.02075967e-02  5.14476240e-01  9.81580615e-01  1.46798468e+00 -1.44933462e-01  3.23302597e-02 -7.98807666e-02 -8.00955296e-01 -5.23091674e-01  2.20382065e-01  2.10842136e-02
   1.49711877e-01  0.00000000e+00  0.00000000e+00 -5.96943200e-01  7.36376822e-01 -1.27137065e-01 -1.11863005e+00  2.38185730e-02 

In [45]:
print(U)

[[-2.76401858e+28 -6.22765811e+28  8.17790716e+28 -5.78348648e+28  9.40688509e+28 -3.22957095e+28 -5.30326124e+27 -9.04893372e+27 -5.41044882e+14 -2.72965828e+27 -1.12655658e+14 -2.72965828e+27 -9.71828490e+27 -9.71828490e+27 -9.12566627e+27 -9.71828490e+27  4.91670485e+27  4.05880227e+27  1.07894194e+27  8.04672477e+27  1.49743378e+27  2.88805602e+28 -5.54152116e+27  2.89543070e+28  1.50998509e+27  1.50998509e+27  1.50998509e+27 -5.53907320e+27 -8.97550742e+27  1.54784062e+27 -3.82774831e+12 -1.16371012e+27]
 [-4.31245297e+27 -7.92882794e+27 -5.30788503e+27 -3.77087516e+27  9.66055515e+27 -1.78543852e+27  1.94294700e+27 -2.53420012e+27 -3.78896269e+13  1.19404151e+27 -3.53598806e+12  1.19404151e+27 -3.15027533e+27 -3.15027533e+27 -4.62806968e+27 -3.15027533e+27  2.63833174e+25  3.94176639e+27  6.65747273e+26  3.96232196e+27  2.11518587e+27  1.77131584e+27  3.52457453e+27  1.99955947e+27  2.62018291e+27  2.62018291e+27  2.62018291e+27  3.55017182e+27 -3.96814995e+27  2.32013304e+27 -5.